# Rozdział 8 — Metody wykorzystujące dodatkową informację

Notebook demonstracyjny do pełnego rozdziału książki. Wszystkie dane są syntetyczne.
Nie wymaga WAV ani pobierania danych. Źródło echa: kontrolowana paczka szumu, nie mowa.
Definicje algorytmów i generatorów znajdują się w src/bspsbook/additional_information.py.
Uruchom z katalogu pakietu lub notebooks/. Zależności: requirements_ch08.txt.
Pełna argumentacja i ograniczenia znajdują się w samodzielnym tekście rozdziału.
Figury zapisujemy w osobnych PNG/SVG. Nie osadzamy obrazów, audio ani widgetów.

Kod pomocniczy odtworzono 11 listopada 2026 r. z kanonicznego M7, paszportów
i opisu książkowego; nie jest kopią bajtową utraconego modułu. Szczegóły
i porównanie wyników: chapter_notes/ch08_reconstruction.md.


In [1]:
from pathlib import Path
import sys,json,platform,importlib.metadata
import numpy as np
import scipy,matplotlib,statsmodels,pywt,PyEMD
from scipy import signal
import matplotlib.pyplot as plt
ROOT=next((p for p in [Path.cwd(),Path.cwd().parent] if (p/'src/bspsbook/additional_information.py').is_file()),None)
if ROOT is None:raise RuntimeError('Uruchom z katalogu pakietu lub notebooks/.')
sys.path.insert(0,str(ROOT/'src'))
from bspsbook.additional_information import *
from bspsbook.plotting import configure_plots,save_figure
configure_plots(chapter=8);FIG=ROOT/'figures/generated';RESULTS={};CHECKS=[];PASSPORTS={}
def check(name,condition):
    if not bool(condition):raise AssertionError(name)
    CHECKS.append(name);print('OK:',name)
def rmse(a,b):return float(np.sqrt(np.mean((np.asarray(a)-np.asarray(b))**2)))
def db(p):return 10*np.log10(np.maximum(p,np.finfo(float).tiny))
def smooth_mse(e,fs=100):return np.convolve(e*e,np.ones(round(.5*fs))/round(.5*fs),'same')
ENV={'python':platform.python_version(),'numpy':np.__version__,'scipy':scipy.__version__,
     'matplotlib':matplotlib.__version__,'statsmodels':statsmodels.__version__,
     'EMD-signal':importlib.metadata.version('EMD-signal'),'PyEMD':PyEMD.__version__,
     'PyWavelets':pywt.__version__,'nbformat':importlib.metadata.version('nbformat'),
     'rng':'numpy.random.default_rng / PCG64',
     'seeds':[2026071202,2026071203,2026071204,2026071205,2026071216,
              2026071207,2026071208,2026071209,2026071210]}
print(json.dumps(ENV,ensure_ascii=False,indent=2))


{
  "python": "3.12.14",
  "numpy": "2.3.5",
  "scipy": "1.17.0",
  "matplotlib": "3.10.8",
  "statsmodels": "0.14.6",
  "EMD-signal": "1.10.0",
  "PyEMD": "1.10.0",
  "PyWavelets": "1.8.0",
  "nbformat": "5.10.4",
  "rng": "numpy.random.default_rng / PCG64",
  "seeds": [
    2026071202,
    2026071203,
    2026071204,
    2026071205,
    2026071216,
    2026071207,
    2026071208,
    2026071209,
    2026071210
  ]
}


## E8.1 — predykcja i reszty; rys. 8.1

M7.2.1: AR(4),fs200 Hz,2400 próbek po500 rozruchowych,seed2026071202.
Bieguny r0.96/24Hz i r0.93/36Hz. Cały zapis centrowany/SD.
Pierwsze700 próbek; Yule–Walker method=mle,demean=True.
ACF reszt lags1–40 z dzielnikiemN; linie±1.96/sqrt(Nres) wyłącznie orientacyjne.

![Rys. 8.1](../figures/generated/fig_08_01_prediction.png)


In [2]:
tar,xar,den_true,fsar=ar_case();x700=xar[:700];fit4=fit_ar(x700,4)
fig,ax=plt.subplots(3,1,figsize=(9.5,7.7),layout='constrained')
tt=np.arange(700)/fsar;z=x700-x700.mean()
ax[0].plot(tt,z,label='Dane');ax[0].plot(tt,fit4['pred'],'--',label='Predykcja AR(4)',lw=.8)
ax[0].set(xlim=(.5,1.2),ylabel='Amplituda [j.u.]',title='Predykcja jednej próbki w przód');ax[0].legend()
ax[1].plot(tt[4:],fit4['residual'],lw=.7);ax[1].set(ylabel='Reszta [j.u.]',title='Reszta dopasowania')
rows=[]
for p in [2,4,8]:
    fit=fit_ar(x700,p);a=acf_biased(fit['residual'],40)
    ax[2].plot(np.arange(1,41)/fsar,a[1:],'o-',ms=2,label=f'p={p}')
    rows.append({'p':p,'sigma2_YW':fit['sigma2'],'empirical_residual_variance':float(np.var(fit['residual']))})
bound=1.96/np.sqrt(len(fit4['residual']))
ax[2].axhline(bound,color='0.5',ls=':');ax[2].axhline(-bound,color='0.5',ls=':')
ax[2].set(xlabel='Opóźnienie [s]',ylabel='ACF [-]',title='Zależności pozostawione w resztach');ax[2].legend(ncol=3)
for a in ax[:2]:a.set_xlabel('Czas [s]')
scaled=fit_ar(3*x700,4)
check('E8.1: skala amplitudy nie zmienia współczynników YW',np.allclose(fit4['a'],scaled['a']))
check('E8.1: wariancja YW skaluje się kwadratowo',np.isclose(scaled['sigma2'],9*fit4['sigma2']))
check('E8.1: znaki konwencji AR i statsmodels',np.allclose(fit4['a'],-fit4['rho']))
check('E8.1: reszty mają N-p próbek',len(fit4['residual'])==696)
check('E8.1: brak predykcji bez historii',np.isnan(fit4['pred'][:4]).all())
RESULTS['E8.1']={'residual_diagnostics':rows,'approximate_ACF_bound':bound}

save_figure(fig,FIG,'fig_08_01_prediction')


OK: E8.1: skala amplitudy nie zmienia współczynników YW
OK: E8.1: wariancja YW skaluje się kwadratowo
OK: E8.1: znaki konwencji AR i statsmodels
OK: E8.1: reszty mają N-p próbek
OK: E8.1: brak predykcji bez historii
Figura: figures/generated/fig_08_01_prediction.png (+ SVG)


## E8.2 — rząd, kryteria i modelowa PSD; rys. 8.2

M7.3.1: ten sam zapis,wycinek[150:550],N400,fs200 Hz.
Rzędy2,4,6,8,12,16,24,30; wspólneN w przybliżonychAIC/BIC.
PSD jednostronna,nfft4096; WelchHann,nperseg128,noverlap64,nfft4096,detrendconstant,density.
Maksima5–80Hz,minimalny odstęp20prążków; największe2 do kontroli stabilności.

![Rys. 8.2](../figures/generated/fig_08_02_ar_order_psd.png)


In [3]:
short=xar[150:550];orders=[2,4,6,8,12,16,24,30];arrows=order_table(short,orders)
fig,ax=plt.subplots(3,1,figsize=(9.5,8.6),layout='constrained')
for p in [4,6,30]:
    f,P=ar_psd(fit_ar(short,p));ax[0].plot(f,db(P),label=f'AR({p})')
fw,Pw=signal.welch(short,200,nperseg=128,noverlap=64,nfft=4096)
ax[0].plot(fw,db(Pw),'--',label='Welch 128',lw=.9)
for freq in [24,36]:ax[0].axvline(freq,color='0.5',ls=':')
ax[0].set(xlim=(5,80),ylim=(-65,0),xlabel='Częstotliwość [Hz]',ylabel='PSD [dB re 1 j.u.²/Hz]');ax[0].legend(ncol=4)
for key in ['AIC','BIC']:ax[1].plot(orders,[r[key] for r in arrows],'o-',label=key)
ax[1].set(ylabel='Kryterium [umowne]',xlabel='Rząd p');ax[1].legend()
ax[2].plot(orders,[r['ACF2'] for r in arrows],'o-')
ax[2].set(xlabel='Rząd p',ylabel='Suma ACF² [-]',title='Diagnostyka reszt nie zastępuje stabilności widma')
f,P=ar_psd(fit_ar(short,4));area=float(P.sum()*(f[1]-f[0]))
check('E8.2: całka PSD odtwarza wariancję modelu YW',np.isclose(area,np.var(short),rtol=1e-10))
check('E8.2: PSD nieujemna i jednostronna',np.all(P>=0) and f[-1]==100)
check('E8.2: bieguny rozważanych YW stabilne',all(r['max_pole']<1 for r in arrows))
check('E8.2: kryteria dla tej realizacji wskazują p6',all(min(arrows,key=lambda r:r[k])['p']==6 for k in ['AIC','BIC']))
table='| p | Wariancja YW | Wariancja reszt | AIC | BIC | Suma ACF² | Maksima 5–80 Hz |\n|---:|---:|---:|---:|---:|---:|---:|\n'
for r in arrows:
    if r['p'] in [2,4,6,30]:
        table+=f"| {r['p']} | {r['sigma2_YW']:.4f} | {r['empirical_residual_variance']:.4f} | {r['AIC']:.1f} | {r['BIC']:.1f} | {r['ACF2']:.3f} | {r['peak_count_5_80']} |\n"
(ROOT/'data/ch08_ar_table.md').write_text(table, encoding='utf-8', newline='\n')
RESULTS['E8.2']={'orders':arrows,'PSD_integral':area,'variance':float(np.var(short))}

save_figure(fig,FIG,'fig_08_02_ar_order_psd')


OK: E8.2: całka PSD odtwarza wariancję modelu YW
OK: E8.2: PSD nieujemna i jednostronna
OK: E8.2: bieguny rozważanych YW stabilne
OK: E8.2: kryteria dla tej realizacji wskazują p6
Figura: figures/generated/fig_08_02_ar_order_psd.png (+ SVG)


## E8.3 — stabilność względem danych; rys. 8.3

M7.3.2: okna400próbek[150:550] i[550:950].
Rzędy2–30 jakE8.2; porównanie2najwyższych maksimów po sortowaniu częstotliwości.
Nie jest to test tożsamości pików. Osobno AR4 dlaN400,700,1200 odindeksu150.
Znany generator służy tylko do kontroli symulacji.

![Rys. 8.3](../figures/generated/fig_08_03_ar_stability.png)


In [4]:
neighbor=xar[550:950];nbrows=order_table(neighbor,orders)
fig,ax=plt.subplots(2,2,figsize=(10,7.5),layout='constrained')
for a,p in zip(ax[0],[6,30]):
    for x,label in [(short,'Okno A'),(neighbor,'Okno B')]:
        f,P=ar_psd(fit_ar(x,p));a.plot(f,db(P),label=label)
    a.set(xlim=(10,55),ylim=(-50,0),xlabel='Częstotliwość [Hz]',ylabel='PSD [dB re 1 j.u.²/Hz]',title=f'AR({p})');a.legend()
stability=[]
for r,s in zip(arrows,nbrows):
    if len(r['top_frequencies_Hz'])==len(s['top_frequencies_Hz'])==2:
        stability.append({'p':r['p'],'shift_Hz':float(np.mean(abs(np.array(r['top_frequencies_Hz'])-s['top_frequencies_Hz'])))})
ax[1,0].plot([r['p'] for r in stability],[r['shift_Hz'] for r in stability],'o-')
ax[1,0].set(xlabel='Rząd p',ylabel='Średnia zmiana maksimów [Hz]')
for n in [400,700,1200]:
    f,P=ar_psd(fit_ar(xar[150:150+n],4));ax[1,1].plot(f,db(P),label=f'N={n}')
ax[1,1].set(xlim=(10,55),ylim=(-50,0),xlabel='Częstotliwość [Hz]',ylabel='PSD [dB re 1 j.u.²/Hz]',title='Ten sam rząd, inny czas obserwacji');ax[1,1].legend()
check('E8.3: ten sam model używa różnych okien danych',not np.array_equal(short,neighbor))
check('E8.3: wysoki rząd tworzy więcej lokalnych maksimów',arrows[-1]['peak_count_5_80']>2)
check('E8.3: wszystkie estymaty skończone',all(np.isfinite(r['shift_Hz']) for r in stability))
RESULTS['E8.3']={'neighbor_orders':nbrows,'sorted_top_two_shift':stability}

save_figure(fig,FIG,'fig_08_03_ar_stability')


OK: E8.3: ten sam model używa różnych okien danych
OK: E8.3: wysoki rząd tworzy więcej lokalnych maksimów
OK: E8.3: wszystkie estymaty skończone
Figura: figures/generated/fig_08_03_ar_stability.png (+ SVG)


## E8.4 — jeden krok odsiewania; rys. 8.4

M7.4.1:6s,fs200Hz,seed2026071203;trend0.45sin(2π0.35t),
chirp5–22Hz a0.75 Tukey0.25;burst34Hz a0.7,t3.2s,σ0.22s;szumSD0.08.
Ekstrema distance3; sześcienne splajny z dodatkowymi kotwicami na końcach.
To ilustracja,nie pełny algorytm PyEMD z lustrzanym przedłużeniem.

![Rys. 8.4](../figures/generated/fig_08_04_sifting.png)


In [5]:
te,xstep=emd_step_case();maxi,mini,upper,lower,mean,proto=one_sift(xstep,te)
fig,ax=plt.subplots(3,1,figsize=(9.5,7.7),sharex=True,layout='constrained')
ax[0].plot(te,xstep,lw=.8);ax[0].plot(te[maxi],xstep[maxi],'.',ms=3);ax[0].plot(te[mini],xstep[mini],'.',ms=3)
ax[0].set(ylabel='Dane [j.u.]',title='Ekstrema i kotwice na końcach')
ax[1].plot(te,xstep,color='0.7',lw=.7)
for a,label in [(upper,'Górna'),(lower,'Dolna'),(mean,'Średnia')]:ax[1].plot(te,a,label=label)
ax[1].set_ylabel('Obwiednie [j.u.]');ax[1].legend(ncol=3)
ax[2].plot(te,proto,lw=.7);ax[2].set(xlabel='Czas [s]',ylabel='Proto-IMF [j.u.]')
check('E8.4: odsiewanie odejmuje średnią obwiedni',np.allclose(proto+mean,xstep))
check('E8.4: splajny przechodzą przez węzły',np.allclose(upper[maxi],xstep[maxi]) and np.allclose(lower[mini],xstep[mini]))
check('E8.4: ekstrema i wartości skończone',len(maxi)>10 and np.isfinite(proto).all())
RESULTS['E8.4']={'mean_envelope_RMS':float(np.sqrt(np.mean(mean**2))),'extrema_distance_samples':3}

save_figure(fig,FIG,'fig_08_04_sifting')


OK: E8.4: odsiewanie odejmuje średnią obwiedni
OK: E8.4: splajny przechodzą przez węzły
OK: E8.4: ekstrema i wartości skończone
Figura: figures/generated/fig_08_04_sifting.png (+ SVG)


## E8.5 — pełna dekompozycja i reszta; rys. 8.5

M7.4.2: te same daneE8.4; EMD-signal1.10.0,cubic,nbsym2,MAX_ITERATION300,
max_imf6; progi i reguły zapisane w paszporcie. Reszta zget_imfs_and_residue().
Nie definiujemy testowanej reszty przez ręczne x−sum(IMF).

![Rys. 8.5](../figures/generated/fig_08_05_imfs.png)


In [6]:
imstep,resstep,settings=decompose(xstep,te,max_imf=6);PASSPORTS['EMD']=settings
fig,ax=plt.subplots(len(imstep)+2,1,figsize=(9.5,1.35*(len(imstep)+2)),sharex=True,layout='constrained')
ax[0].plot(te,xstep,lw=.65);ax[0].set_ylabel('Dane [j.u.]')
for a,i,z in zip(ax[1:-1],range(len(imstep)),imstep):a.plot(te,z,lw=.65);a.set_ylabel(f'IMF{i+1}\n[j.u.]')
ax[-1].plot(te,resstep,lw=.8);ax[-1].set(xlabel='Czas [s]',ylabel='Reszta [j.u.]')
reconstruction=imstep.sum(axis=0)+resstep
check('E8.5: rekonstrukcja z obiektów biblioteki',np.allclose(reconstruction,xstep,atol=1e-11))
check('E8.5: reszta nie została zgubiona',np.linalg.norm(resstep)>1e-6)
check('E8.5: IMF odrębne od reszty',imstep.ndim==2 and imstep.shape[1]==len(resstep))
RESULTS['E8.5']={'IMF_count':len(imstep),'max_reconstruction_error':float(max(abs(reconstruction-xstep))),
                 'residue_norm':float(np.linalg.norm(resstep))}

save_figure(fig,FIG,'fig_08_05_imfs')


OK: E8.5: rekonstrukcja z obiektów biblioteki
OK: E8.5: reszta nie została zgubiona
OK: E8.5: IMF odrębne od reszty
Figura: figures/generated/fig_08_05_imfs.png (+ SVG)


## E8.6 — wybór modów według celu; rys. 8.6

M7.5.1:8s,fs200Hz,seed2026071204;slow0.35sin(2π0.45t),
b1=Gaussian(t2.2,σ0.7)*chirp7–16Hz,b2=0.8Gaussian(t5.7,σ0.38)*sin29Hz;
szumSD0.12. EMD jakE8.5,max_imf7. DominantaWelch512,Hann,50%overlap.
Selekcja dominant5–40Hz; alternatywa dodaje sąsiedni wolniejszy mod.
DWTdb4,level5,symmetric,soft; sigma=median(abs(cD1))/0.6745,lambda=sigma*sqrt(2logN).
Szybkie odniesienie b1+b2; pełne slow+b1+b2. Brzegi po10%próbek.

![Rys. 8.6](../figures/generated/fig_08_06_selected_reconstruction.png)


In [7]:
td,clean,noisy,slow,b1,b2=emd_selection_case()
imsel,ressel,settings7=decompose(noisy,td,max_imf=7);PASSPORTS['EMD_selection']=settings7
summary=characterize(imsel,noisy);selected,fast=select_fast(imsel,summary);fastref=b1+b2
nextslower=min(set(range(len(imsel)))-set(selected)) if len(selected)<len(imsel) else None
alternative=fast+(imsel[nextslower] if nextslower is not None else 0)
fullsel=[i for i,r in enumerate(summary) if r['dominant_Hz']<=40]
emdfull=imsel[fullsel].sum(axis=0)+ressel
coeffs=pywt.wavedec(noisy,'db4',level=5,mode='symmetric')
sigma=np.median(abs(coeffs[-1]))/.6745;threshold=sigma*np.sqrt(2*np.log(len(noisy)))
dwt=pywt.waverec([coeffs[0]]+[pywt.threshold(c,threshold,'soft') for c in coeffs[1:]],'db4',mode='symmetric')[:len(noisy)]
fig,ax=plt.subplots(3,1,figsize=(9.5,8),sharex=True,layout='constrained')
ax[0].plot(td,fastref,label='Odniesienie szybkie',lw=1.3)
ax[0].plot(td,fast,label='Wybrane IMF',lw=.7);ax[0].plot(td,alternative,'--',label='Z dodatkowym modem',lw=.6)
ax[0].set_ylabel('Szybkie [j.u.]');ax[0].legend(ncol=3)
ax[1].plot(td,clean,label='Pełne odniesienie',lw=1.3);ax[1].plot(td,emdfull,label='EMD',lw=.7);ax[1].plot(td,dwt,label='DWT',lw=.7)
ax[1].set_ylabel('Pełne [j.u.]');ax[1].legend(ncol=3)
ax[2].plot(td,fast-fastref,lw=.65)
ax[2].axvspan(0,.8,color='0.5',alpha=.15);ax[2].axvspan(7.2,8,color='0.5',alpha=.15)
ax[2].set(xlabel='Czas [s]',ylabel='Błąd szybki [j.u.]')
check('E8.6: suma raportowanych udziałów IMF równa1',np.isclose(sum(r['fraction_of_IMF_energy_sum'] for r in summary),1))
check('E8.6: pełna dekompozycja odtwarza wejście',np.allclose(imsel.sum(axis=0)+ressel,noisy,atol=1e-11))
check('E8.6: pełne i szybkie odniesienie różnią się trendem',np.allclose(clean-fastref,slow))
check('E8.6: DWT zachowuje liczbę próbek',len(dwt)==len(clean))
check('E8.6: alternatywa rzeczywiście zmienia zestaw modów',nextslower is not None and not np.allclose(fast,alternative))
RESULTS['E8.6']={'IMFs':summary,'selected_1based':[i+1 for i in selected],
    'alternative_added_IMF':None if nextslower is None else nextslower+1,
    'fast_RMSE':regional_errors(fast,fastref),'alternative_fast_RMSE':regional_errors(alternative,fastref),
    'full_RMSE_EMD':rmse(emdfull,clean),'full_RMSE_DWT':rmse(dwt,clean),
    'DWT_sigma':float(sigma),'DWT_threshold':float(threshold)}

save_figure(fig,FIG,'fig_08_06_selected_reconstruction')


OK: E8.6: suma raportowanych udziałów IMF równa1
OK: E8.6: pełna dekompozycja odtwarza wejście
OK: E8.6: pełne i szybkie odniesienie różnią się trendem
OK: E8.6: DWT zachowuje liczbę próbek
OK: E8.6: alternatywa rzeczywiście zmienia zestaw modów
Figura: figures/generated/fig_08_06_selected_reconstruction.png (+ SVG)


## E8.7 — mieszanie modów i otoczenie; rys. 8.7

M7.5.2:6s,fs200Hz,seed2026071205;A chirp7–22Hz,a0.75;
B chirp24–9Hz,a0.65,przerwa2.75–3.15s,szumSD0.04. EMDmax_imf6.
Kontrola otoczenia: wycinek1–7s tych samych danychE8.6,ta sama selekcja5–40Hz.
Korelacje ze źródłami znane wyłącznie w symulacji.

![Rys. 8.7](../figures/generated/fig_08_07_mode_mixing.png)


In [8]:
tmix,xmix,ca,cb=emd_mixing_case();immix,rmix,_=decompose(xmix,tmix,max_imf=6)
correlations=np.corrcoef(np.vstack([immix,ca,cb]))[:-2,-2:]
fig,ax=plt.subplots(4,1,figsize=(9.5,8.4),sharex=True,layout='constrained')
ax[0].plot(tmix,ca,label='Składowa A',lw=.7);ax[0].plot(tmix,cb,label='Składowa B',lw=.7);ax[0].legend(ncol=2)
ax[0].set_ylabel('Źródła [j.u.]')
for a,i,z in zip(ax[1:],range(3),immix[:3]):
    a.plot(tmix,z,lw=.65);a.set_ylabel(f'IMF{i+1} [j.u.]')
    a.set_title(f'corr(A)={correlations[i,0]:.3f}; corr(B)={correlations[i,1]:.3f}',fontsize=9)
for a in ax:a.axvspan(2.75,3.15,color='0.5',alpha=.15)
ax[-1].set_xlabel('Czas [s]')
crop=(td>=1)&(td<7);ic,rc,_=decompose(noisy[crop],td[crop],max_imf=7)
cropsummary=characterize(ic,noisy[crop]);cropsel,cropfast=select_fast(ic,cropsummary)
check('E8.7: rekonstrukcja mimo mieszania',np.allclose(immix.sum(axis=0)+rmix,xmix,atol=1e-11))
check('E8.7: pierwszy IMF współdzieli oba źródła',np.all(abs(correlations[0])>.4))
check('E8.7: wycinek zachowuje te same próbki',len(cropfast)==crop.sum())
RESULTS['E8.7']={'IMF_correlations_A_B':correlations.tolist(),'crop_IMF_1based':[i+1 for i in cropsel],
    'crop_vs_full_output_RMSE':rmse(cropfast,fast[crop]),
    'crop_fast_errors':regional_errors(cropfast,fastref[crop]),
    'full_context_errors_same_crop':regional_errors(fast[crop],fastref[crop])}

save_figure(fig,FIG,'fig_08_07_mode_mixing')


OK: E8.7: rekonstrukcja mimo mieszania
OK: E8.7: pierwszy IMF współdzieli oba źródła
OK: E8.7: wycinek zachowuje te same próbki
Figura: figures/generated/fig_08_07_mode_mixing.png (+ SVG)


## E8.8 — znany wzorzec i czas detekcji; rys. 8.8

M7.6.1:5s,fs500Hz,seed2026071216;wzorzec120próbek,38Hz,Gaussianσ25ms.
Środki0.8/1.75/2.9/4.15s,amplitudy1/0.8/1.2/0.65;szumSD0.65.
Korelacjavalid;próg3SD zjawnie znanego odcinka szumu:środkipod.55s.
Odstępdetekcji.5s,tolerancja±50ms;kalibracja pełnegosplotu bezszumu.

![Rys. 8.8](../figures/generated/fig_08_08_matched_detection.png)


In [9]:
pat=template();tm,xm,truth,starts=matched_case(pat);score=scores(xm,pat)
noise_region=score['time']<.55;threshold3=3*np.std(score['raw'][noise_region])
detected,ix=detect(score['raw'],score['time'],threshold3);matched=match_events(detected,truth)
fig,ax=plt.subplots(3,1,figsize=(9.5,8),layout='constrained')
ax[0].plot((np.arange(len(pat))-60)/500,pat);ax[0].set(xlabel='Czas względem środka [s]',ylabel='Wzorzec [j.u.]')
ax[1].plot(tm,xm,lw=.7)
for t in truth:ax[1].axvline(t,color='0.5',ls=':')
ax[1].set(xlabel='Czas [s]',ylabel='Dane [j.u.]')
ax[2].plot(score['time'],score['raw'],lw=.8);ax[2].plot(detected,score['raw'][ix],'o',label='Detekcje')
ax[2].axhline(threshold3,color='0.5',ls='--',label='Próg 3 SD')
ax[2].set(xlabel='Czas środka [s]',ylabel='Korelacja [j.u.²]');ax[2].legend()
calibration=np.zeros(600);calstart=123;calibration[calstart:calstart+len(pat)]=pat
full=signal.convolve(calibration,pat[::-1],mode='full')
kmax=int(np.argmax(full));estimated_start=kmax-(len(pat)-1)
check('E8.8: korelacja zgodna ze splotem odwróconego wzorca',np.allclose(score['raw'],signal.convolve(xm,pat[::-1],mode='valid')))
check('E8.8: kalibracja indeksu początku',estimated_start==calstart)
check('E8.8: kalibracja środka',(estimated_start+60)/500==(calstart+60)/500)
check('E8.8: poprawny odcinek kalibracji szumu',score['time'][noise_region][-1]+.12<truth[0]-.12)
check('E8.8: cztery wykrycia bez pominięć dla tej realizacji',matched['TP']==4 and matched['FP']==matched['FN']==0)
RESULTS['E8.8']={'threshold':float(threshold3),'detections_s':detected.tolist(),'metrics':matched,
    'calibration_start':calstart,'calibration_kmax':kmax,'estimated_start':estimated_start}

save_figure(fig,FIG,'fig_08_08_matched_detection')


OK: E8.8: korelacja zgodna ze splotem odwróconego wzorca
OK: E8.8: kalibracja indeksu początku
OK: E8.8: kalibracja środka
OK: E8.8: poprawny odcinek kalibracji szumu
OK: E8.8: cztery wykrycia bez pominięć dla tej realizacji
Figura: figures/generated/fig_08_08_matched_detection.png (+ SVG)


## E8.9 — niedopasowanie i próg; rys. 8.9

M7.6.2: te same dane i zdarzenia; szerokość wzorca×0.8/1/1.2.
Porównanie znormalizowanym iloczynem skalarnym,bez centrowania lokalnego.
Maksima w±.12s wokół znanych zdarzeń są diagnostyką symulacji.
Osobno progi2/3/4SD na surowej korelacji;wspólny odcinek szumu,odstęp.5s,tolerancja±.05s.

![Rys. 8.9](../figures/generated/fig_08_09_template_limits.png)


In [10]:
mismatch=[];thresholdrows=[]
fig,ax=plt.subplots(2,1,figsize=(9.5,6.8),layout='constrained')
for scale in [.8,1.,1.2]:
    temp=template(width=.025*scale);ss=scores(xm,temp);values=[]
    for t in truth:values.append(float(np.max(ss['normalized'][abs(ss['time']-t)<=.12])))
    mismatch.append({'scale':scale,'norm':float(np.linalg.norm(temp)),'scores':values,'mean':float(np.mean(values))})
    ax[0].plot(np.arange(1,5),values,'o-',label=f'Szerokość ×{scale}')
ax[0].set(xlabel='Numer znanego zdarzenia',ylabel='Znormalizowany wynik [-]',xticks=[1,2,3,4]);ax[0].legend(ncol=3)
for factor in [2,3,4]:
    threshold=factor*np.std(score['raw'][noise_region]);dt,ii=detect(score['raw'],score['time'],threshold)
    metrics=match_events(dt,truth);thresholdrows.append({'factor':factor,'threshold':float(threshold),**metrics})
ax[1].plot([r['factor'] for r in thresholdrows],[r['FP'] for r in thresholdrows],'o-',label='FP')
ax[1].plot([r['factor'] for r in thresholdrows],[r['FN'] for r in thresholdrows],'s--',label='FN')
ax[1].set(xlabel='Próg [SD korelacji szumu]',ylabel='Liczba zdarzeń',xticks=[2,3,4]);ax[1].legend()
check('E8.9: normy wzorców różnią się',len(set(round(r['norm'],5) for r in mismatch))==3)
check('E8.9: wynik znormalizowany ograniczony przez1',all(max(abs(np.array(r['scores'])))<=1+1e-12 for r in mismatch))
for r in thresholdrows:check(f"E8.9: TP+FN przy progu {r['factor']}",r['TP']+r['FN']==len(truth))
RESULTS['E8.9']={'mismatch':mismatch,'threshold_sweep':thresholdrows}

save_figure(fig,FIG,'fig_08_09_template_limits')


OK: E8.9: normy wzorców różnią się
OK: E8.9: wynik znormalizowany ograniczony przez1
OK: E8.9: TP+FN przy progu 2
OK: E8.9: TP+FN przy progu 3
OK: E8.9: TP+FN przy progu 4
Figura: figures/generated/fig_08_09_template_limits.png (+ SVG)


## E8.10 — kanał odniesienia i zmiana toru; rys. 8.10

M7.7.1:24s,fs100Hz,seed2026071207;syntetyczny PPG-podobny zparGaussów:
od.6s co.82s,σ45ms i70ms,drugi opóźniony140ms,a.35;centrowanie/SD.
Źródło artefaktuAR1=.96,skalowanieSD;referencja+szumSD.08;główny+szumSD.05.
h1[.55,−.25,.15,.08],h2[.25,.35,−.18,.12],zmiana12s.
FIR12;Wiener ridge1e−6 uczony wyłącznie na czasie<12s;LMSμ.002,h0=0.
Ocena wspólna na próbkach odindeksu11;wygładzenie błędu prostokątne.5s.

![Rys. 8.10](../figures/generated/fig_08_10_reference_cancellation.png)


In [11]:
tl,s,d,ref,artifact,source,h1,h2=reference_case();M=12
w=wiener_cancel(d,ref,tl,length=M);l=lms_cancel(d,ref,length=M,mu=.002)
ta=w['time'];s2=s[M-1:];d2=d[M-1:]
metrics={name:filter_metrics(z,s2) for name,z in [('Raw',d2),('Wiener',w['output']),('LMS',l['output'])]}
fig,ax=plt.subplots(3,1,figsize=(9.5,8),layout='constrained')
ax[0].plot(tl,d,label='Kanał główny',lw=.7);ax[0].plot(tl,s,label='Odniesienie czyste',lw=.8)
ax[0].set(xlim=(10,14),ylabel='Amplituda [j.u.]');ax[0].legend(ncol=2)
for z,label in [(w['output'],'Wiener'),(l['output'],'LMS'),(s2,'Czysty')]:ax[1].plot(ta,z,label=label,lw=.8)
ax[1].set(xlim=(10,14),ylabel='Wynik [j.u.]');ax[1].legend(ncol=3)
for z,label in [(d2,'Surowy'),(w['output'],'Wiener'),(l['output'],'LMS')]:ax[2].plot(ta,smooth_mse(z-s2),label=label,lw=.8)
ax[2].set(ylabel='Lokalny MSE [j.u.²]',xlim=(0,24));ax[2].legend(ncol=3)
for a in ax:a.axvline(12,color='0.5',ls='--');a.set_xlabel('Czas [s]')
check('E8.10: trening kończy się przed zmianą toru',ta[w['train']].max()<12 and ta[~w['train']].min()==12)
check('E8.10: rozwiązanie regularizowanych równań normalnych',np.allclose(w['R']@w['h'],w['r']))
check('E8.10: wynik jest błędem d-y',np.allclose(w['output']+w['y'],d2))
check('E8.10: LMS skończony bez przerwania',l['stopped_at_row'] is None and np.isfinite(l['output']).all())
check('E8.10: MSE obydwu metod niższy od wejścia',metrics['Wiener']['MSE']<metrics['Raw']['MSE'] and metrics['LMS']['MSE']<metrics['Raw']['MSE'])
check('E8.10: Wiener lepszy w tej realizacji',metrics['Wiener']['MSE']<metrics['LMS']['MSE'])
RESULTS['E8.10']={'all_metrics':metrics,'settled_after15s':{name:filter_metrics(z,s2,ta>=15) for name,z in [('Raw',d2),('Wiener',w['output']),('LMS',l['output'])]},
 'Wiener_h':w['h'].tolist(),'train_samples':int(w['train'].sum()),'condition_number_regularized_R':w['condition_number']}

save_figure(fig,FIG,'fig_08_10_reference_cancellation')


OK: E8.10: trening kończy się przed zmianą toru
OK: E8.10: rozwiązanie regularizowanych równań normalnych
OK: E8.10: wynik jest błędem d-y
OK: E8.10: LMS skończony bez przerwania
OK: E8.10: MSE obydwu metod niższy od wejścia
OK: E8.10: Wiener lepszy w tej realizacji
Figura: figures/generated/fig_08_10_reference_cancellation.png (+ SVG)


## E8.11 — sygnał pożądany w referencji; rys. 8.11

M7.7.2:referencja zE8.10 +0.35s[n],wszystkie inne dane iμ.002 bez zmian.
J=średniae²;MSE=średnia(e−s)²;G=<e,s>/<s,s> — opis projekcji,nie estymata fizjologii.
Pełny wspólny czas;rysunek pokazuje fragment15–18s i miary całości.

![Rys. 8.11](../figures/generated/fig_08_11_contaminated_reference.png)


In [12]:
badref=ref+.35*s;bad=lms_cancel(d,badref,length=M,mu=.002)
badmetrics=filter_metrics(bad['output'],s2);goodmetrics=filter_metrics(l['output'],s2)
fig,ax=plt.subplots(2,1,figsize=(9.5,6.8),layout='constrained')
for z,label in [(s2,'Czysty PPG-podobny'),(l['output'],'Dobra referencja'),(bad['output'],'Domieszka PPG w referencji')]:ax[0].plot(ta,z,label=label,lw=.8)
ax[0].set(xlim=(15,18),xlabel='Czas [s]',ylabel='Amplituda [j.u.]');ax[0].legend(ncol=2)
indices=np.arange(3)
ax[1].bar(indices-.18,[goodmetrics[k] for k in ['J','MSE','G']],.36,label='Dobra referencja')
ax[1].bar(indices+.18,[badmetrics[k] for k in ['J','MSE','G']],.36,label='Z domieszką PPG')
ax[1].set(xticks=indices,xticklabels=['J [j.u.²]','MSE [j.u.²]','G [-]'],ylabel='Wartość',title='Mniejszy koszt nie dowodzi lepszego zachowania sygnału');ax[1].legend()
check('E8.11: zła referencja obniża koszt J',badmetrics['J']<goodmetrics['J'])
check('E8.11: zła referencja zwiększa właściwy błąd',badmetrics['MSE']>goodmetrics['MSE'])
check('E8.11: projekcja na czysty sygnał maleje',badmetrics['G']<goodmetrics['G'])
RESULTS['E8.11']={'good_reference':goodmetrics,'contaminated_reference':badmetrics,
 'reference_clean_correlation':float(np.corrcoef(badref,s)[0,1])}

save_figure(fig,FIG,'fig_08_11_contaminated_reference')


OK: E8.11: zła referencja obniża koszt J
OK: E8.11: zła referencja zwiększa właściwy błąd
OK: E8.11: projekcja na czysty sygnał maleje
Figura: figures/generated/fig_08_11_contaminated_reference.png (+ SVG)


## E8.12 — krok, skala i kierunek opóźnień; rys. 8.12

Kontrole implementacji M7.7.1:μ.0002/.002/.02; te same dane.
Skala referencjiβ10,ten samμ.002 iμ/β²=.00002.
Diagnostyczne przerwanie jeśli||h||>1e6:pozostałe wynikiNaN,bez obcinania wag.
Osobna kalibracja:biały sygnałseed2026071210,tor.7z^(−3),FIR12,ridge1e−6.

![Rys. 8.12](../figures/generated/fig_08_12_lms_step_scale.png)


In [13]:
muruns=[(mu,lms_cancel(d,ref,M,mu)) for mu in [.0002,.002,.02]]
scaled_bad=lms_cancel(d,10*ref,M,.002);scaled_good=lms_cancel(d,10*ref,M,.002/100)
scalematch=rmse(scaled_good['output'],l['output'])
fig,ax=plt.subplots(2,1,figsize=(9.5,7),layout='constrained')
for mu,run in muruns:ax[0].plot(ta,smooth_mse(run['output']-s2),label=f'μ={mu}',lw=.8)
ax[0].set(xlabel='Czas [s]',ylabel='Lokalny MSE [j.u.²]');ax[0].legend(ncol=3)
ax[1].semilogy(ta,np.linalg.norm(l['weights'],axis=1),label='Referencja ×1, μ=.002')
ax[1].semilogy(ta,np.linalg.norm(scaled_bad['weights'],axis=1),label='Referencja ×10, μ=.002')
ax[1].semilogy(ta,np.linalg.norm(scaled_good['weights'],axis=1),label='Referencja ×10, μ=.00002')
ax[1].set(xlabel='Czas [s]',ylabel='Norma wag [-]');ax[1].legend(fontsize=7)
rng=np.random.default_rng(2026071210);u=rng.normal(size=1000);hd=np.r_[0.,0.,0.,.7]
delayed=signal.lfilter(hd,[1],u);wc=wiener_cancel(delayed,u,np.arange(1000)/100,12,12.)
check('E8.12: skala wejścia wymaga odwrotnego kwadratu kroku',scalematch<1e-12)
check('E8.12: wagi skalują się odwrotnie',np.allclose(scaled_good['weights'],l['weights']/10,atol=1e-12))
check('E8.12: ten sam krok przy skali10 uruchamia diagnostykę',scaled_bad['stopped_at_row'] is not None)
check('E8.12: nie symulujemy wyników po przerwaniu',np.isnan(scaled_bad['output'][scaled_bad['stopped_at_row']:]).all())
check('E8.12: wektor opóźnień ma poprawny kierunek',np.argmax(abs(wc['h']))==3 and abs(wc['h'][3]-.7)<1e-4)
RESULTS['E8.12']={'mu_sweep':[{'mu':mu,**filter_metrics(run['output'],s2)} for mu,run in muruns],
 'scale_beta':10,'scale_corrected_output_RMSE':scalematch,'guard_stop_row':scaled_bad['stopped_at_row'],
 'guard_stop_time_s':float(ta[scaled_bad['stopped_at_row']]),'delay_control_h':wc['h'].tolist()}

save_figure(fig,FIG,'fig_08_12_lms_step_scale')


OK: E8.12: skala wejścia wymaga odwrotnego kwadratu kroku
OK: E8.12: wagi skalują się odwrotnie
OK: E8.12: ten sam krok przy skali10 uruchamia diagnostykę
OK: E8.12: nie symulujemy wyników po przerwaniu
OK: E8.12: wektor opóźnień ma poprawny kierunek
Figura: figures/generated/fig_08_12_lms_step_scale.png (+ SVG)


## E8.13 — pełny splot i cepstrum echa; rys. 8.13

Zastępstwo brakującego WAV zM7.8.1:fs16000Hz,N16000,seed2026071208.
Źródło=0.25biały szum×Gaussian(t.35s,σ.025s);nie mowa. EchoD1360=85ms,g.55.
Pełny splot liniowy,Nobs17360,NFFT32768,bez kadrowania iokna.
Cepstrumlogmodułu naturalnego,epsilon1e−10;szukanie25–180ms,modułmaksimum.

![Rys. 8.13](../figures/generated/fig_08_13_echo_cepstrum.png)


In [14]:
ts,source_echo,fse=echo_source();observed,h,D=linear_echo(source_echo,fse)
nfft=32768;cobs,Xobs=real_cepstrum(observed,nfft);csource,Xsource=real_cepstrum(source_echo,nfft)
ch,H=real_cepstrum(h,nfft);peak,delay=estimate_delay(cobs,fse)
q=np.arange(nfft)/fse;freq=np.fft.fftfreq(nfft,1/fse);truthpad=np.pad(source_echo,(0,len(observed)-len(source_echo)))
fig,ax=plt.subplots(3,1,figsize=(9.5,8),layout='constrained')
ax[0].plot(np.arange(len(observed))/fse,observed,label='Z echem',lw=.6)
ax[0].plot(ts,source_echo,label='Źródło',lw=.6);ax[0].set(xlim=(.25,.53),xlabel='Czas [s]',ylabel='Amplituda [j.u.]');ax[0].legend()
positive=(freq>=0)&(freq<=1000)
ax[1].plot(freq[positive],np.log(abs(Xobs[positive])+1e-10),label='Obserwacja')
ax[1].plot(freq[positive],np.log(abs(Xsource[positive])+1e-10),alpha=.6,label='Źródło')
ax[1].set(xlabel='Częstotliwość [Hz]',ylabel='ln modułu [umowne]');ax[1].legend()
mask=(q>=.025)&(q<=.18)
for c,label in [(cobs,'Obserwacja'),(csource,'Źródło'),(ch,'Znany tor')]:ax[2].plot(1000*q[mask],c[mask],label=label,lw=.8)
ax[2].axvline(delay*1000,color='0.5',ls=':');ax[2].set(xlabel='Quefrency [ms]',ylabel='Cepstrum [umowne]');ax[2].legend(ncol=3)
check('E8.13: pełna długość splotu',len(observed)==len(source_echo)+len(h)-1)
check('E8.13: iloczyn DFT odpowiada pełnemu splotowi',np.allclose(Xobs,Xsource*H,atol=1e-11))
check('E8.13: indeks maksimum równy opóźnieniu',peak==D==1360)
check('E8.13: współczynnik cepstrum toru g/2',np.isclose(ch[D],.55/2,atol=1e-7))
check('E8.13: krok quefrency zależy od fs',np.isclose(q[1]-q[0],1/fse))
RESULTS['E8.13']={'delay_samples':D,'estimated_delay_s':delay,'source_samples':len(source_echo),
 'observed_samples':len(observed),'nfft':nfft,'channel_cepstrum_at_D':float(ch[D]),
 'observed_cepstrum_at_D':float(cobs[D]),'regularized_additivity_max_error':float(max(abs(cobs-csource-ch)))}

save_figure(fig,FIG,'fig_08_13_echo_cepstrum')


OK: E8.13: pełna długość splotu
OK: E8.13: iloczyn DFT odpowiada pełnemu splotowi
OK: E8.13: indeks maksimum równy opóźnieniu
OK: E8.13: współczynnik cepstrum toru g/2
OK: E8.13: krok quefrency zależy od fs
Figura: figures/generated/fig_08_13_echo_cepstrum.png (+ SVG)


## E8.14 — lifter, faza i czułość; rys. 8.14

M7.8.2:te same daneE8.13. Symetryczna prostokątna maska wokół±D:
półszerokość4ms,współczynnik.15,poza1;faza obserwacji zachowana.
Kontrola3epsilon(1e−12/−10/−8)×2NFFT(32768/65536)×3szerokości(2/4/8ms).
Ocena na17360próbkach,czysty sygnał dopełnionyzerami;osobnoenergia poza tym wsparciem.
Znany moduł czystego źródła +faza obserwacji służą tylko diagnostyce fazy.

![Rys. 8.14](../figures/generated/fig_08_14_lifter_limits.png)


In [15]:
lift=lift_echo(observed,fse,nfft)
oracle=np.fft.ifft(abs(Xsource)*np.exp(1j*np.angle(Xobs))).real[:len(observed)]
sensitivity=[]
for eps in [1e-12,1e-10,1e-8]:
    for nf in [32768,65536]:
        for width in [.002,.004,.008]:
            run=lift_echo(observed,fse,nf,eps,width)
            sensitivity.append({'epsilon':eps,'nfft':nf,'half_width_ms':width*1000,
                'delay_ms':run['delay_s']*1000,'RMSE':rmse(run['output'],truthpad),
                'relative_error_energy':float(np.sum((run['output']-truthpad)**2)/np.sum(truthpad**2)),
                'energy_outside_support_relative':float(np.sum(run['full_output'][len(observed):]**2)/np.sum(truthpad**2))})
fig,ax=plt.subplots(3,1,figsize=(9.5,8),layout='constrained')
mask=(q>=.07)&(q<=.10)
ax[0].plot(1000*q[mask],cobs[mask],label='Przed');ax[0].plot(1000*q[mask],lift['modified'][mask],label='Po lifterze')
ax[0].set(xlabel='Quefrency [ms]',ylabel='Cepstrum [umowne]');ax[0].legend()
timeobs=np.arange(len(observed))/fse
for z,label in [(observed,'Z echem'),(lift['output'],'Po lifterze'),(truthpad,'Czyste odniesienie')]:ax[1].plot(timeobs,z,label=label,lw=.7)
ax[1].set(xlim=(.40,.50),xlabel='Czas [s]',ylabel='Amplituda [j.u.]');ax[1].legend(ncol=3)
for nf in [32768,65536]:
    rows=[r for r in sensitivity if r['epsilon']==1e-10 and r['nfft']==nf]
    ax[2].plot([r['half_width_ms'] for r in rows],[r['RMSE'] for r in rows],'o-',label=f'NFFT={nf}')
ax[2].axhline(rmse(observed,truthpad),color='0.5',ls=':',label='Bez liftera')
ax[2].set(xlabel='Półszerokość liftera [ms]',ylabel='RMSE [j.u.]');ax[2].legend()
check('E8.14: lifter ma symetrię rzeczywistego cepstrum',np.array_equal(lift['mask'][1:],lift['mask'][:0:-1]))
check('E8.14: rekonstrukcja rzeczywista',lift['imaginary_max']<1e-12)
check('E8.14: zachowana faza obserwacji',np.max(abs(np.angle(lift['Y']*np.conj(lift['X']))))<1e-10)
check('E8.14: RMSE w tym przykładzie zmniejszony',rmse(lift['output'],truthpad)<rmse(observed,truthpad))
check('E8.14: poprawny moduł przy złej fazie nie daje czystego źródła',rmse(oracle,truthpad)>1e-4)
check('E8.14: komplet18wariantów',len(sensitivity)==18)
check('E8.14: czułość nie zmienia wykrytego opóźnienia',all(r['delay_ms']==85 for r in sensitivity))
RESULTS['E8.14']={'RMSE_before':rmse(observed,truthpad),'RMSE_after':rmse(lift['output'],truthpad),
 'RMSE_clean_magnitude_observed_phase':rmse(oracle,truthpad),'sensitivity':sensitivity}

save_figure(fig,FIG,'fig_08_14_lifter_limits')


OK: E8.14: lifter ma symetrię rzeczywistego cepstrum
OK: E8.14: rekonstrukcja rzeczywista
OK: E8.14: zachowana faza obserwacji
OK: E8.14: RMSE w tym przykładzie zmniejszony
OK: E8.14: poprawny moduł przy złej fazie nie daje czystego źródła
OK: E8.14: komplet18wariantów
OK: E8.14: czułość nie zmienia wykrytego opóźnienia
Figura: figures/generated/fig_08_14_lifter_limits.png (+ SVG)


## E8.15 — pik bez echa; rys. 8.15

Kontrprzykład doM7.8.3:fs16000Hz,seed2026071209,brak kanału z echem.
Blok640próbek(40ms):biały szum×Hannperiodic,powtórzony25razy,+szumSD.005.
Ta sama reguła wyszukiwania25–180ms,epsilon1e−10,NFFT32768.
To okresowe źródło syntetyczne,nie nagranie mowy.

![Rys. 8.15](../figures/generated/fig_08_15_periodicity_without_echo.png)


In [16]:
tp,periodic,fsp,period=periodic_source();cp,Xp=real_cepstrum(periodic,32768)
ip,qp=estimate_delay(cp,fsp);qperiod=np.arange(len(cp))/fsp
fig,ax=plt.subplots(2,1,figsize=(9.5,6.7),layout='constrained')
ax[0].plot(tp,periodic,lw=.65);ax[0].set(xlim=(0,.16),xlabel='Czas [s]',ylabel='Amplituda [j.u.]',title='Okresowe źródło bez echa')
sel=(qperiod>=.025)&(qperiod<=.18)
ax[1].plot(qperiod[sel]*1000,cp[sel],lw=.8);ax[1].axvline(qp*1000,color='0.5',ls=':',label=f'Maksimum {qp*1000:.0f} ms')
ax[1].set(xlabel='Quefrency [ms]',ylabel='Cepstrum [umowne]');ax[1].legend()
check('E8.15: pik okresowości wykryty bez echa',ip==period==640)
check('E8.15: ta sama reguła użyta w obydwu przykładach',.025<=qp<=.18)
check('E8.15: źródło ma pełne1s',len(periodic)==16000)
RESULTS['E8.15']={'echo_added':False,'source_period_samples':period,'detected_index':ip,'detected_quefrency_s':qp}

save_figure(fig,FIG,'fig_08_15_periodicity_without_echo')


OK: E8.15: pik okresowości wykryty bez echa
OK: E8.15: ta sama reguła użyta w obydwu przykładach
OK: E8.15: źródło ma pełne1s
Figura: figures/generated/fig_08_15_periodicity_without_echo.png (+ SVG)


## Zapis wyników i paszportów

JSON zapisuje wyniki liczbowe i nazwy asercji. Wersje środowiska są raportem wykonania,
nie dowodem uruchomienia protokołu jądra Jupyter. Skrypt run_ch08.py wykonuje zwykłe
komórki w świeżym procesie; --jupyter wymaga dostępnej komunikacji lokalnej.

W bieżącym środowisku gniazda TCP i IPC są niedostępne. Kontrolowane wykonanie
skryptowe nie jest testem interfejsu Run all Jupyter; nie używamy atrap bibliotek.


In [17]:
PASSPORTS.update({
 'AR':{'fs_Hz':200,'seed':2026071202,'burnin':500,'samples':2400,'YW_method':'mle, demean=True',
       'orders':orders,'nfft':4096,'AIC_BIC_common_N':400,'peak_min_distance_bins':20},
 'EMD_step_data':{'seed':2026071203,'fs_Hz':200,'samples':1200,'extrema_distance_illustration':3},
 'EMD_selection_data':{'seed':2026071204,'fs_Hz':200,'samples':1600,'dominant_Welch_nperseg':512,'selected_band_Hz':[5,40]},
 'EMD_mixing_data':{'seed':2026071205,'fs_Hz':200,'samples':1200,'gap_s':[2.75,3.15]},
 'DWT_comparator':{'wavelet':'db4','level':5,'mode':'symmetric','threshold':'soft, sigma median(abs(cD1))/.6745 * sqrt(2logN)'},
 'matched':{'fs_Hz':500,'seed':2026071216,'samples':2500,'template_samples':120,'carrier_Hz':38,'sigma_s':.025,
     'noise_SD':.65,'valid_centers_noise_calibration_s':'<0.55','separation_s':.5,'event_tolerance_s':.05},
 'reference':{'fs_Hz':100,'seed':2026071207,'samples':2400,'length':12,'mu':.002,'ridge':1e-6,
     'training_time_s':'<12','change_s':12,'mu_sweep':[.0002,.002,.02],'beta':10,'guard_norm':1e6,
     'h1':h1.tolist(),'h2':h2.tolist()},
 'echo':{'fs_Hz':16000,'seed':2026071208,'source_samples':16000,'source':'Gaussian noise packet, not speech',
     'echo_gain':.55,'delay_s':.085,'epsilon':1e-10,'nfft':32768,'search_s':[.025,.18],
     'half_width_s':.004,'lifter_attenuation':.15,'window':None,'convolution':'full linear','output_samples':17360},
 'periodic_noecho':{'seed':2026071209,'period_samples':640,'noise_SD':.005},
 'delay_calibration':{'seed':2026071210,'delay_samples':3,'gain':.7}})
RESULTS['audit']={'passed_checks':len(CHECKS),'checks':CHECKS}
for path,obj in [('data/ch08_results.json',RESULTS),('data/ch08_parameter_passports.json',PASSPORTS),
                 ('chapter_notes/ch08_environment.json',ENV)]:
    (ROOT/path).write_text(json.dumps(obj,ensure_ascii=False,indent=2,allow_nan=False)+'\n', encoding='utf-8', newline='\n')
print('Asercje:',len(CHECKS));print('Eksperymenty:15; figury:15 par PNG/SVG')

# Kopia wejść i prawdy odniesienia; obliczenia powyżej nie odczytują tego pliku.
np.savez_compressed(ROOT/'data/ch08_synthetic_inputs.npz',
    ar_time_s=tar, ar_signal=xar, ar_denominator=den_true, ar_fs_Hz=fsar,
    emd_step_time_s=te, emd_step_signal=xstep,
    emd_selection_time_s=td, emd_selection_clean=clean, emd_selection_noisy=noisy,
    emd_selection_slow=slow, emd_selection_burst1=b1, emd_selection_burst2=b2,
    emd_mixing_time_s=tmix, emd_mixing_signal=xmix, emd_mixing_component_A=ca, emd_mixing_component_B=cb,
    matched_time_s=tm, matched_signal=xm, matched_template=pat, matched_centers_s=truth, matched_starts_samples=starts,
    reference_time_s=tl, reference_clean=s, reference_primary=d, reference_channel=ref,
    reference_artifact=artifact, reference_source=source, reference_h1=h1, reference_h2=h2,
    echo_source_time_s=ts, echo_source=source_echo, echo_observed=observed, echo_channel=h, echo_fs_Hz=fse,
    echo_delay_samples=D, periodic_time_s=tp, periodic_signal=periodic, periodic_fs_Hz=fsp, periodic_period_samples=period)
print('Dane syntetyczne i prawda odniesienia: data/ch08_synthetic_inputs.npz')


Asercje: 65
Eksperymenty:15; figury:15 par PNG/SVG
Dane syntetyczne i prawda odniesienia: data/ch08_synthetic_inputs.npz
